In [2]:
from pyspark.sql import functions as F

silver_df = spark.table("silver.meter_readings")

dim_date_df = spark.table("gold.dim_date")
dim_time_df = spark.table("gold.dim_time")
dim_tariff_df = spark.table("gold.dim_tariff")

print("Demand pattern fact build initialised.")


StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 4, Finished, Available, Finished, False)

Demand pattern fact build initialised.


In [3]:
demand_base_df = (
    silver_df
    .withColumn(
        "ReadingMinute",
        F.minute("ReadingTimestamp")
    )
    .withColumn(
        "TimeKey",
        (
            F.hour("ReadingTimestamp") * 100
            + F.col("ReadingMinute")
        ).cast("int")
    )
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 5, Finished, Available, Finished, False)

In [4]:
print("SOURCE MINUTE VALUES")
print("-" * 50)

demand_base_df.groupBy(
    "ReadingMinute"
).count().orderBy(
    "ReadingMinute"
).show()

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 6, Finished, Available, Finished, False)

SOURCE MINUTE VALUES
--------------------------------------------------
+-------------+--------+
|ReadingMinute|   count|
+-------------+--------+
|            0|83906707|
|           30|83904754|
+-------------+--------+



In [5]:
demand_agg_df = (
    demand_base_df
    .groupBy(
        "ReadingDate",
        "TimeKey",
        "TariffType"
    )
    .agg(
        F.sum("ConsumptionKWh").alias("TotalConsumptionKWh"),
        F.avg("ConsumptionKWh").alias("AverageConsumptionKWh"),
        F.max("ConsumptionKWh").alias("PeakHouseholdConsumptionKWh"),
        F.count("*").alias("ReadingCount"),
        F.countDistinct("HouseholdID").alias("DistinctHouseholds")
    )
)

print(
    f"Demand pattern aggregated rows: "
    f"{demand_agg_df.count():,}"
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 7, Finished, Available, Finished, False)

Demand pattern aggregated rows: 79,454


In [6]:
demand_with_date_df = (
    demand_agg_df
    .join(
        dim_date_df.select(
            "DateKey",
            "Date"
        ),
        demand_agg_df["ReadingDate"] == dim_date_df["Date"],
        how="left"
    )
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 8, Finished, Available, Finished, False)

In [7]:
demand_with_tariff_df = (
    demand_with_date_df
    .join(
        dim_tariff_df,
        on="TariffType",
        how="left"
    )
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 9, Finished, Available, Finished, False)

In [8]:
demand_with_keys_df = (
    demand_with_tariff_df
    .join(
        dim_time_df.select("TimeKey"),
        on="TimeKey",
        how="left"
    )
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 10, Finished, Available, Finished, False)

In [9]:
fact_demand_df = (
    demand_with_keys_df
    .select(
        "DateKey",
        "TimeKey",
        "TariffKey",
        "TotalConsumptionKWh",
        "AverageConsumptionKWh",
        "PeakHouseholdConsumptionKWh",
        "ReadingCount",
        "DistinctHouseholds"
    )
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 11, Finished, Available, Finished, False)

In [10]:
key_validation_df = (
    fact_demand_df
    .agg(
        F.sum(
            F.when(F.col("DateKey").isNull(), 1).otherwise(0)
        ).alias("MissingDateKeys"),

        F.sum(
            F.when(F.col("TimeKey").isNull(), 1).otherwise(0)
        ).alias("MissingTimeKeys"),

        F.sum(
            F.when(F.col("TariffKey").isNull(), 1).otherwise(0)
        ).alias("MissingTariffKeys")
    )
)

display(key_validation_df)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e55e9a19-da34-4e62-a9cc-a290c3f02f3b)

In [11]:
duplicate_demand_keys = (
    fact_demand_df
    .groupBy(
        "DateKey",
        "TimeKey",
        "TariffKey"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    f"Duplicate demand fact keys: "
    f"{duplicate_demand_keys:,}"
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 13, Finished, Available, Finished, False)

Duplicate demand fact keys: 0


In [12]:
silver_consumption = (
    silver_df
    .agg(
        F.sum("ConsumptionKWh").alias("TotalConsumption")
    )
    .collect()[0]["TotalConsumption"]
)

demand_consumption = (
    fact_demand_df
    .agg(
        F.sum("TotalConsumptionKWh").alias("TotalConsumption")
    )
    .collect()[0]["TotalConsumption"]
)

difference = silver_consumption - demand_consumption

print("DEMAND FACT CONSUMPTION RECONCILIATION")
print("-" * 50)

print(
    f"Silver consumption: "
    f"{silver_consumption:,.6f}"
)

print(
    f"Demand Gold consumption: "
    f"{demand_consumption:,.6f}"
)

print(
    f"Difference: "
    f"{difference:,.12f}"
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 14, Finished, Available, Finished, False)

DEMAND FACT CONSUMPTION RECONCILIATION
--------------------------------------------------
Silver consumption: 35,539,823.306396
Demand Gold consumption: 35,539,823.306385
Difference: 0.000010527670


In [13]:
display(
    fact_demand_df
    .orderBy(
        "DateKey",
        "TimeKey",
        "TariffKey"
    )
    .limit(30)
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 434f687d-d756-465e-8cf6-79a0b179a1fb)

In [14]:
(
    fact_demand_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.fact_demand_pattern")
)

print(
    "gold.fact_demand_pattern "
    "written successfully."
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 16, Finished, Available, Finished, False)

gold.fact_demand_pattern written successfully.


In [15]:
persisted_demand_df = spark.table(
    "gold.fact_demand_pattern"
)

print("FACT DEMAND PATTERN")
print("-" * 50)

print(
    f"Rows: "
    f"{persisted_demand_df.count():,}"
)

persisted_demand_df.printSchema()

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 17, Finished, Available, Finished, False)

FACT DEMAND PATTERN
--------------------------------------------------
Rows: 79,454
root
 |-- DateKey: integer (nullable = true)
 |-- TimeKey: integer (nullable = true)
 |-- TariffKey: integer (nullable = true)
 |-- TotalConsumptionKWh: double (nullable = true)
 |-- AverageConsumptionKWh: double (nullable = true)
 |-- PeakHouseholdConsumptionKWh: double (nullable = true)
 |-- ReadingCount: long (nullable = true)
 |-- DistinctHouseholds: long (nullable = true)



In [16]:
spark.sql(
    "DESCRIBE DETAIL gold.fact_demand_pattern"
).select(
    "numFiles",
    "sizeInBytes",
    "partitionColumns"
).show(truncate=False)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 18, Finished, Available, Finished, False)

+--------+-----------+----------------+
|numFiles|sizeInBytes|partitionColumns|
+--------+-----------+----------------+
|17      |2569370    |[]              |
+--------+-----------+----------------+



In [17]:
from pyspark.sql import functions as F

demand_fact = spark.table(
    "gold.fact_demand_pattern"
)

tariff_schedule = (
    spark.table("gold.tariff_schedule")
    .select(
        "DateKey",
        "TimeKey",
        "TariffBand"
    )
)

enriched_demand_fact = (
    demand_fact.alias("f")
    .join(
        tariff_schedule.alias("s"),
        (
            (F.col("f.DateKey") == F.col("s.DateKey"))
            &
            (F.col("f.TimeKey") == F.col("s.TimeKey"))
        ),
        "left"
    )
    .select(
        "f.*",
        F.coalesce(
            F.col("s.TariffBand"),
            F.lit("Not Applicable")
        ).alias("TariffBand")
    )
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 19, Finished, Available, Finished, False)

In [18]:
print("DEMAND FACT ENRICHMENT")
print("-" * 50)

print(
    f"Original rows: "
    f"{demand_fact.count():,}"
)

print(
    f"Enriched rows: "
    f"{enriched_demand_fact.count():,}"
)

duplicate_keys = (
    enriched_demand_fact
    .groupBy(
        "DateKey",
        "TimeKey",
        "TariffKey"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    f"Duplicate fact keys: "
    f"{duplicate_keys:,}"
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 20, Finished, Available, Finished, False)

DEMAND FACT ENRICHMENT
--------------------------------------------------
Original rows: 79,454
Enriched rows: 79,454
Duplicate fact keys: 0


In [19]:
(
    enriched_demand_fact
    .groupBy("TariffBand")
    .count()
    .orderBy(F.desc("count"))
    .show()
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 21, Finished, Available, Finished, False)

+--------------+-----+
|    TariffBand|count|
+--------------+-----+
|Not Applicable|44414|
|        Normal|30144|
|           Low| 3320|
|          High| 1576|
+--------------+-----+



In [20]:
null_tariff_bands = (
    enriched_demand_fact
    .filter(F.col("TariffBand").isNull())
    .count()
)

print(
    f"Null TariffBand rows: "
    f"{null_tariff_bands:,}"
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 22, Finished, Available, Finished, False)

Null TariffBand rows: 0


In [21]:
original_consumption = (
    demand_fact
    .agg(
        F.sum("TotalConsumptionKWh")
        .alias("Total")
    )
    .collect()[0]["Total"]
)

enriched_consumption = (
    enriched_demand_fact
    .agg(
        F.sum("TotalConsumptionKWh")
        .alias("Total")
    )
    .collect()[0]["Total"]
)

print(
    f"Original consumption: "
    f"{original_consumption:.6f}"
)

print(
    f"Enriched consumption: "
    f"{enriched_consumption:.6f}"
)

print(
    f"Difference: "
    f"{original_consumption - enriched_consumption:.12f}"
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 23, Finished, Available, Finished, False)

Original consumption: 35539823.306385
Enriched consumption: 35539823.306385
Difference: 0.000000000000


In [22]:
enriched_demand_fact.cache()

enriched_demand_fact.count()

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 24, Finished, Available, Finished, False)

79454

In [23]:
(
    enriched_demand_fact
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "gold.fact_demand_pattern"
    )
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 25, Finished, Available, Finished, False)

In [24]:
final_demand = spark.table(
    "gold.fact_demand_pattern"
)

print(
    f"Persisted rows: "
    f"{final_demand.count():,}"
)

final_demand.printSchema()

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 26, Finished, Available, Finished, False)

Persisted rows: 79,454
root
 |-- DateKey: integer (nullable = true)
 |-- TimeKey: integer (nullable = true)
 |-- TariffKey: integer (nullable = true)
 |-- TotalConsumptionKWh: double (nullable = true)
 |-- AverageConsumptionKWh: double (nullable = true)
 |-- PeakHouseholdConsumptionKWh: double (nullable = true)
 |-- ReadingCount: long (nullable = true)
 |-- DistinctHouseholds: long (nullable = true)
 |-- TariffBand: string (nullable = true)



In [25]:
(
    spark.table("gold.fact_demand_pattern")
    .groupBy("TariffBand")
    .count()
    .orderBy(F.desc("count"))
    .show()
)

StatementMeta(, 4d8f05b8-d459-4409-afe6-93d18a9c4043, 27, Finished, Available, Finished, False)

+--------------+-----+
|    TariffBand|count|
+--------------+-----+
|Not Applicable|44414|
|        Normal|30144|
|           Low| 3320|
|          High| 1576|
+--------------+-----+

